# Import


In [3]:
#from odf.opendocument import OpenDocumentSpreadsheet
#from odf.table import Table, TableRow, TableCell
#from odf.text import P
import pandas as pd
from docx import Document
from docx.shared import Pt, RGBColor, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
import os


# Globals

In [8]:
base_path= "/home/gaia/Desktop/MIPA/Data_Catalogue_Qualitative/"
data_catalog= f"{base_path}/Data_catalog v.3.xlsx"
csv_file= os.path.join(base_path, "Data_catalog v.csv")

# Utils & Functions


In [13]:
def file_to_docx(input_file, output_file, separator=','):
    """
    Converte CSV o Excel in DOCX con una pagina per ogni riga
    """
    # Determina il tipo di file dall'estensione
    file_estensione = os.path.splitext(input_file)[1].lower()
    
    # Leggi il file in base all'estensione
    if file_estensione in ['.xlsx', '.xls']:
        # File Excel
        df = pd.read_excel(input_file)
        print(f"File: {input_file} caricato")
    else:
        # File CSV (default)
        df = pd.read_csv(input_file, sep=separator)
        print(f"File {input_file} caricato")
    
    print(f"Ci sono {len(df.columns)} variabili nella tabella di {len(df)} fonti")
    
    # eliminiamo colonne vuote e sistemiamo i nomi delle variabili (rimuovendo eventuali note tra parentesi)
    df = df.dropna(axis=1, how='all')
    df.columns = [col.replace('(menu a tendina)', '').strip() for col in df.columns]

    # Crea documento
    doc = Document()
    # configura margini e stile
    for section in doc.sections:
        section.top_margin = Cm(2)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(2.5)
        section.right_margin = Cm(2.5)
    style = doc.styles['Title']
    style.font.size = Pt(22)
    style.font.bold = True
    style.font.color.rgb = RGBColor(0, 64, 128)  
    
    # Crea pagine
    for index, row in df.iterrows():
        # Titolo pagina (prima colonna)
        titolo = str(row.iloc[0]) if len(row) > 0 else f"Pagina {index+1}"
        doc.add_heading(titolo, level=1)
        doc.add_paragraph()  # Spazio
        
        # Contenuto (dalla seconda colonna in poi, per evitare duplicati)
        for col_name in df.columns[1:]:  # Salta la prima colonna (già usata come titolo)
            valore = row[col_name]
            valore_str = "" if pd.isna(valore) else str(valore)
            
            # Sostituisci \n con spazio
            valore_str = valore_str.replace('\n', ' ').replace('\r', ' ')
            
            # Crea paragrafo
            p = doc.add_paragraph()
            
            # Aggiungi nome campo in grassetto
            p.add_run(f"{col_name}: ").bold = True
            
            # Aggiungi valore in NORMALE
            p.add_run(valore_str)
            
            # Spazio tra variabili
            p.paragraph_format.space_after = Pt(12)
        
        # Aggiungi interruzione di pagina SOLO se non è l'ultima riga
        if index < len(df) - 1:
            doc.add_page_break()
    
    # Salva
    doc.save(output_file)
    print(f"Create {len(df)} pagine in {output_file}")

# Run

In [14]:

# ESECUZIONE
if __name__ == "__main__":
    # Per file CSV:
    # file_to_docx("tuo_file.csv", "output_da_csv.docx")
    
    # Per file Excel:
    file_to_docx(data_catalog, "catalogodati_da_excel.docx")
    
    # Oppure usa le funzioni specifiche:
    # excel_to_docx("dataset_catalog.xlsx", "catalogodati.xlsx.docx")
    # csv_to_docx("dataset_catalog.csv", "catalogodati.csv.docx")

File: /home/gaia/Desktop/MIPA/Data_Catalogue_Qualitative//Data_catalog v.3.xlsx caricato
Ci sono 14 variabili nella tabella di 28 fonti
Create 28 pagine in catalogodati_da_excel.docx


/home/gaia/.local/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
